# 02 · How does a computer *read* a sentence?

In Notebook 01 an image turned into 512 numbers. Now it's the text's turn. We'll follow a few real news articles through CLIP's text side and watch a string of letters become a vector.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 3, Section 3.1, *Foundations of Neural Language Models*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. Which articles are we following?

Just like with the image in Notebook 01, it's way easier to understand a model when you can point at the exact examples going in. So let's pick our articles first.

### 1.1 Grab our tools

Same toolbox as before: `datasets` for the articles, `transformers` for CLIP's tokenizer and text model, PyTorch for the tensors, and Matplotlib for a couple of plots.

In [ ]:
import matplotlib.pyplot as plt  # plotting, tensors, datasets, and CLIP
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

Importing PyTorch doesn't put anything on the GPU by itself. We pick the device once here and reuse it every time we move the model or token tensors.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise

### 1.2 Where do the articles come from?

We'll use [AG News](https://huggingface.co/datasets/fancyzhx/ag_news), a dataset of short news texts, each labeled with one of four broad topics. Why real articles instead of a few sentences we make up? Because real text is messy. It has punctuation, names, and unusual words, and those are exactly the things that make tokenization interesting.

<img src="../assets/diagrams/notebook2/dataset.png" alt="Fields and topic labels in AG News" width="800">

Along with the dataset name, we also pin a **revision hash**. Think of it as a commit ID for the dataset: it points at one exact snapshot. If someone updates the dataset later, we still get the same rows.

In [ ]:
DATASET_ID = "fancyzhx/ag_news"  # AG News on the Hugging Face Hub
DATASET_REVISION = "eb185aade064a813bc0b7f42de02595523103ca4"  # pin one exact snapshot so the rows never change

The tokenizer and the text model come from the same CLIP checkpoint we used in Notebook 01. That pairing really matters here. A token ID is just a number, and it only means something to the vocabulary and the weights that learned it.

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"  # tokenizer and text model must come from the same checkpoint

Time to load the data. We grab the whole training split for now, so we can go looking for examples of each topic. The first run downloads it; after that it comes from your local cache.

In [ ]:
news = load_dataset(  # downloads once, then loads from the cache
    DATASET_ID,
    split="train",
    revision=DATASET_REVISION,
)

The labels are stored as integers, but the dataset also remembers which topic name goes with each integer. Let's hang on to that mapping.

In [ ]:
label_feature = news.features["label"]  # knows how to turn label integers into topic names
label_names = label_feature.names

Quick sanity check. You should see the columns (`text` and `label`), the number of rows, and four topic names.

Why bother? Because label `2` means nothing by itself. It only means "Business" or "Sports" or whatever *this* dataset says it means. Reading the mapping first saves you from a really sneaky kind of mistake.

In [ ]:
print(news)  # columns and row count
print("topics:", label_names)

### 1.3 Pick two articles per topic

We want a balanced little batch: two articles from each topic. That'll make the comparison at the end much easier to read.

Rather than hard-coding row numbers, let's ask the data. First, find where each topic shows up for the first time.

In [ ]:
all_labels = list(news["label"])  # every label, in dataset order
first_indices = [all_labels.index(label) for label in range(len(label_names))]  # first row of each topic

Next, find a *second* article for each topic, and interleave them: topic 0 article 1, topic 0 article 2, topic 1 article 1, and so on. We'll keep this exact order all the way through tokenization and plotting.

In [ ]:
second_indices = [all_labels.index(label, first + 1) for label, first in enumerate(first_indices)]  # next row of the same topic
study_indices = [index for pair in zip(first_indices, second_indices) for index in pair]  # interleave: t0, t0, t1, t1, ...

`select` gives us a tiny 8-row dataset with the same `text` and `label` columns. Small enough to look at, but every row is a real article.

In [ ]:
study = news.select(study_indices)  # our 8 articles: two per topic

Let's peek at them. Each line shows the row number, the topic, and the first 100 characters of the article. Row 0 is our running example, but all eight will get a vector in the end.

In [ ]:
for row_number, row in enumerate(study):  # row number, topic name, start of the article
    topic = label_feature.int2str(row["label"])
    print(f"{row_number}: {topic:8s} | {row['text'][:100]}…")

## 2. How does CLIP chop text into pieces?

A transformer reads a sequence of tokens. For images, CLIP made tokens by cutting the picture into patches. For text, the job falls to a **tokenizer**: it splits a string into pieces from a fixed list (the **vocabulary**) and swaps each piece for its number in that list.

CLIP's tokenizer uses **byte-pair encoding (BPE)**. The idea: common words get to be a single piece, while rarer words get split into smaller chunks that the vocabulary *does* have. So there's never a word it can't handle. It just might take a few pieces.

### 2.1 Load the processor and the model

Same two pieces as in Notebook 01: the processor (which holds the tokenizer) and the model (which holds the weights).

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)  # includes the tokenizer that belongs to this checkpoint

Now the model. Move it to our device and call `.eval()`, so training-only stuff like dropout stays off. Its weights won't change at all in this notebook.

In [ ]:
model = CLIPModel.from_pretrained(MODEL_ID).to(device)  # the pretrained weights
model.eval()  # we're only looking, not training: turn off dropout

The processor carries the tokenizer inside it, so let's pull that out. We also ask the model how long a sequence its text side can take. Reading it from the config beats typing in a number and hoping it matches.

In [ ]:
tokenizer = processor.tokenizer  # the text half of the processor
max_length = model.config.text_config.max_position_embeddings  # CLIP-B/32's text side has 77 positions

The next cell prints which tokenizer class we got, the vocabulary size, and the length limit, which should be 77.

> **Heads up:** The vocabulary size is how many *different* pieces the tokenizer knows. It's not the number of words in an article, and it's not a length limit.

In [ ]:
print("tokenizer:", type(tokenizer).__name__)  # which tokenizer class we actually got
print("vocabulary size:", tokenizer.vocab_size, "| maximum length:", max_length)

CLIP also uses a few **special tokens**, pieces that aren't really part of the text:

- **BOS** (*beginning of sequence*): goes at the very start.
- **EOS** (*end of sequence*): goes right after the last real piece.
- **PAD** (*padding*): filler that makes short texts as long as the longest one, so they can share one rectangular tensor.

Let's see what IDs this checkpoint gives them.

In [ ]:
print(  # special-token IDs, straight from the tokenizer config
    "BOS:", tokenizer.bos_token_id,
    "| EOS:", tokenizer.eos_token_id,
    "| PAD:", tokenizer.pad_token_id,
)

Notice anything odd? PAD and EOS have the **same ID**. That's not a typo. This checkpoint reuses the end-of-text token as its padding token. It'll matter in Section 3.1, when we go looking for where each article ends: we'll need the *first* EOS in each row, since every padding position after it looks like EOS too.

### 2.2 Zoom in on one article's pieces

Let's take the text of row 0 and ask the tokenizer to split it. Nothing fancy yet: no special tokens, no padding, no model. Just the raw split.

CLIP's tokenizer uses **byte-pair encoding** (BPE), from [Sennrich et al. (2016)](https://arxiv.org/abs/1508.07909). The whole learning algorithm fits in a few lines of Python, and the paper prints it:

<img src="../assets/diagrams/notebook2/bpe_algorithm1.png" alt="Algorithm 1 from Sennrich et al.: learning BPE merge operations in Python" width="420">

*Algorithm 1 and Figure 1 from Sennrich et al. (2016).*

The idea: start with every word split into single characters, plus an end-of-word marker (`</w>` in the code, `·` in the little table). Count which pair of neighboring symbols shows up most often, merge it into one new symbol, and repeat. The table under the code shows the first four merges on a toy dictionary: `r·`, then `lo`, then `low`, then `er·`. CLIP learned its merges the same way, just on far more text, ending up with a vocabulary of 49,152 pieces. Common words become a single piece, and rare words get split into several.

In [ ]:
first_text = study[0]["text"]  # our running example
first_pieces = tokenizer.tokenize(first_text)  # split it with CLIP's BPE rules

We'll print the first 180 characters and the first 25 pieces so things stay readable. The total at the end counts *all* the pieces, with nothing cut off.

> **Pause and predict:** Skim the source text for punctuation, a name, or an unusual word. Will it be one piece or several?

In [ ]:
print("source text:", first_text[:180], "…")  # source characters vs. learned pieces
print("first 25 BPE pieces:", first_pieces[:25], "| total:", len(first_pieces))

A few things to notice:

- **Everything is lowercase.** CLIP's tokenizer lowercases text before splitting it, so "Reuters" and "reuters" look exactly the same to the model.
- **Many pieces end in `</w>`.** That marks the end of a word. A piece without it is the start or middle of a longer word, so you can see where a word got split.
- **Punctuation usually gets its own piece**, and less common words break into chunks.

**TODO:** Try the tokenizer on your own text. Made-up words, names, numbers, and typos are all fair game. Which ones stay in one piece?

In [ ]:
# TODO: type anything here and re-run
my_text = "Scientists unveil a hyperspectral telescope in Reykjavik"
my_pieces = tokenizer.tokenize(my_text)
print(len(my_pieces), "pieces:", my_pieces)

Words the vocabulary knows well usually come out as one piece. Rare names and long technical words tend to come apart.

### 2.3 How do we stack eight articles of different lengths?

The model wants one rectangular tensor, but our eight articles all have different lengths. One tokenizer call sorts that out:

- it adds the **BOS** and **EOS** special tokens to every article,
- **truncation** chops off anything past 77 positions,
- **padding** fills shorter rows with PAD until they match the longest row,
- and it builds an **attention mask**: a 1 for every real position the model should read, a 0 for every padding position it should ignore.

In [ ]:
study_texts = list(study["text"])  # keep the dataset's row order
token_batch = tokenizer(
    study_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
).to(device)  # IDs and mask go where the model is

The next cell prints the shape of `input_ids` and `attention_mask`, and the sum of each row's mask. Since the mask is 1 for every real position, its sum is how many positions in that row are real (special tokens included).

> **Pause and predict:** Will both tensors be exactly 77 wide? Or could they be narrower? *Hint: `padding=True` only pads up to the longest row in the batch.*

In [ ]:
print("input IDs:", tuple(token_batch["input_ids"].shape))  # [batch, sequence], never wider than 77
print("attention mask:", tuple(token_batch["attention_mask"].shape))
print("visible lengths:", token_batch["attention_mask"].sum(dim=1).tolist())

Both tensors have the same `[batch, sequence]` shape: 8 rows, and a width set by the longest article, capped at 77. So the width is 77 only if some article hit the limit. Different visible lengths just mean the articles were different lengths to begin with.

Now, a list of integers is great for a model but hard for us to read. Let's turn the first 20 IDs of row 0 back into pieces, to check that the order survived and the special tokens landed where we expect.

In [ ]:
preview_ids = token_batch["input_ids"][0, :20].tolist()  # first 20 IDs of row 0, from the real batch
preview_pieces = tokenizer.convert_ids_to_tokens(preview_ids)  # back to readable pieces

Read the two lists side by side. Each ID is basically an address in the vocabulary, and the piece is what's stored at that address. You should see the BOS token in the first slot, followed by the same pieces we saw in Section 2.2.

> **Heads up:** The IDs don't carry meaning by themselves. Two IDs that are close together aren't "similar" in any useful way. They're just neighbors in a list.

In [ ]:
print("first twenty IDs:", preview_ids)  # numbers on top...
print("matching pieces:", preview_pieces)  # ...readable pieces below

Finally, let's decode row 0 all the way back into text, skipping the special tokens. This is exactly what the model gets to read. If the article was longer than 77 positions, this is what survived truncation, and anything past it simply doesn't exist for CLIP.

In [ ]:
decoded_text = tokenizer.decode(token_batch["input_ids"][0], skip_special_tokens=True)  # what actually reaches the model
print("decoded model text:", decoded_text)

It's all lowercase, matching the pieces we saw earlier. Compare it with the original text in Section 2.2: did anything get lost?

## 3. How do token pieces become one article vector?

### 3.1 Run the text transformer

Right now each position is just an ID, a lookup in the vocabulary. The word "bank" gets the same ID whether it's a river bank or a money bank.

The transformer fixes that. As it runs, each position mixes in information from the others, and comes out as a **contextual state**: a vector that depends on the surrounding text, not just on which token it is.

One difference from the vision side: CLIP's text transformer uses a **causal mask**. Each position can only look at itself and the positions *before* it, never ahead. Keep that in mind; it explains a choice CLIP makes in a moment.

If you've seen the Transformer diagram from [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), CLIP's text tower is essentially a stack of its right-hand blocks, minus the middle attention block (there's no other input to look at, only the text). The **Masked Multi-Head Attention** box is the causal mask we just talked about.

<img src="../assets/diagrams/notebook7/transformer_fig1.png" alt="The Transformer model architecture" width="360">

*Figure 1 from Vaswani et al. (2017). We'll come back to that middle attention block in Notebook 07.*

In [ ]:
with torch.inference_mode():  # no gradients needed, we're just reading features
    text_outputs = model.text_model(
        input_ids=token_batch["input_ids"],
        attention_mask=token_batch["attention_mask"],  # 0s tell the model to skip the padding
    )

Like in Notebook 01, the model hands back a few views of its result. We'll keep two:

- `last_hidden_state`: one contextual state for every position in every article
- `pooler_output`: one summary per article, taken from that article's **EOS position** (after a final layer norm)

Why the EOS position? Because of the causal mask. EOS is the last real position, so it's the only one that got to look at the entire article. On the vision side, CLIP read its summary from a class token at the *front*; here, the summary lives at the *end*.

This comes straight from the [CLIP paper](https://arxiv.org/abs/2103.00020) (Radford et al., 2021). In the part where they describe the text encoder, they say the text is wrapped in start and end tokens, and *"the activations of the highest layer of the transformer at the [EOS] token are treated as the feature representation of the text."* The same paragraph mentions the lowercased BPE vocabulary of 49,152 tokens and a cap on text length, which is why our tokenizer stops at 77 positions.

In [ ]:
token_states = text_outputs.last_hidden_state  # every position, after attention
pooled_text = text_outputs.pooler_output  # the EOS position's summary of each article

> **Pause and predict:** In Notebook 01, the vision states were 768 numbers wide. What shapes will we get here for `token_states` and `pooled_text`? *Hint: the text side has its own width.*

In [ ]:
print("token states [B, L, D]:", tuple(token_states.shape))  # one vector per position, per article
print("pooled text [B, D]:", tuple(pooled_text.shape))  # one vector per article

`[8, sequence length, 512]` for the token states, and `[8, 512]` for the summaries. The text transformer is **512** wide, not 768 like the vision transformer.

Notice that pooling drops the *sequence* axis but keeps the *batch* axis. All eight articles are still there, in the same order as our study rows.

Let's find where each article's EOS actually sits. Remember, PAD has the same ID as EOS, so we look for the **first** match in each row. That's the real end of the article, before any padding.

> **Something to chew on:** All eight rows have the same tensor width. So why can their EOS positions be different? And how should these positions line up with the visible lengths from Section 2.3?

In [ ]:
eos_positions = token_batch["input_ids"].eq(tokenizer.eos_token_id).int().argmax(dim=1)  # argmax finds the first EOS, before any padding
print("end-of-text positions:", eos_positions.tolist())

The width is shared, but the *content* isn't. Shorter articles hit EOS earlier, and everything after it is padding. Each EOS position should be exactly one less than that row's visible length, because positions count from zero and EOS is the last real position. These are the positions `pooler_output` read its summaries from.

### 3.2 Step into the shared image–text space

Our summaries still live in the text transformer's own space. Same two final steps as on the vision side:

1. **Project:** CLIP's learned text projection maps each summary into the shared space. Here it goes from 512 to 512, so the size doesn't change, but the layer is still doing real work: it lines text up with the image vectors from Notebook 01.
2. **Normalize:** scale each vector to length 1 (*L2 normalization*).

In [ ]:
with torch.inference_mode():
    projected_text = model.text_projection(pooled_text)  # CLIP's learned 512 -> 512 text projection
article_vectors = F.normalize(projected_text, dim=-1)  # length 1, so dot product = cosine similarity

You should see one 512-number row per article, and lengths super close to `1.0`. The rows are still in our interleaved order from Section 1.3, which will make the comparison matrix easy to read.

In [ ]:
print("article vectors [B, D]:", tuple(article_vectors.shape))  # 8 rows, same order as the study articles
print("vector lengths:", article_vectors.norm(dim=-1))

### 3.3 Peek inside a few token states

Before we compare articles, let's look at what those contextual states actually look like. The full tensor is way too big to eyeball, so we'll show a small window: the real (non-padding) positions of row 0, at most 20 of them, and only the first 32 of the 512 numbers.

In [ ]:
valid_length = int(token_batch["attention_mask"][0].sum().item())  # skip the padding
display_length = min(valid_length, 20)  # TODO: show more or fewer positions, then re-run the next two cells

For the row labels, we convert the same IDs the model saw back into pieces. That way the labels can't drift out of line with the states.

In [ ]:
state_slice = token_states[0, :display_length, :32].cpu()  # a small window into row 0's states
piece_labels = tokenizer.convert_ids_to_tokens(token_batch["input_ids"][0, :display_length].tolist())  # labels from the real IDs

In the heatmap, look at how *busy* each row is. Pretty much every coordinate has some nonzero value. That's a big difference from a token ID: the ID is one address in a list, while the contextual state is a whole vector shaped by the rest of the article.

> **Heads up:** Same warning as with the image vector. Don't try to read a fixed meaning into one column. These numbers work as a team, and a single coordinate can play different roles in different contexts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))  # one row per token position
heatmap = ax.imshow(state_slice, aspect="auto", cmap="coolwarm")
ax.set_yticks(range(display_length), piece_labels)
ax.set_xlabel("first 32 hidden coordinates")
ax.set_ylabel("CLIP token")
ax.set_title("Contextual token states for one AG News article")
fig.colorbar(heatmap, ax=ax, label="activation")
plt.tight_layout()
plt.show()

## 4. Which articles does CLIP think are alike?

Here's where unit length pays off. Since every article vector has length 1, multiplying the batch by its own transpose gives us **every pairwise cosine similarity** in one go. Row `i`, column `j` is how similar article `i` is to article `j`.

In [ ]:
similarity = article_vectors @ article_vectors.T  # [8, 512] @ [512, 8] -> [8, 8] cosine scores
print("similarity matrix:", tuple(similarity.shape))

8 × 8, one score for every pair. To label the plot, we combine each row's topic name with its article number within that topic (1 or 2), in the same order as the vectors.

In [ ]:
labels = [f"{label_feature.int2str(row['label'])} {i % 2 + 1}" for i, row in enumerate(study)]  # e.g. topic name + 1 or 2

In the heatmap, darker blue means a higher score. We pin the color scale from 0 to 1 so the colors mean the same thing every time you run it.

> **Pause and predict:** The diagonal compares each article with itself. What score will it have? And off the diagonal, do you expect the two articles from the same topic to stand out?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))  # fixed 0-to-1 color scale for every pair
heatmap = ax.imshow(similarity.cpu(), vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
ax.set_title("CLIP cosine similarity between AG News articles")
fig.colorbar(heatmap, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

The diagonal is 1 (give or take a tiny rounding error), since every vector points the same way as itself. The off-diagonal cells are the interesting part: darker cells are pairs of articles CLIP placed closer together.

> **Heads up:** Treat this as an observation, not a test of how good CLIP is at sorting news. CLIP was trained to match images with captions, not to classify news topics, and eight hand-picked articles are way too few to back up any general claim.

## Recap

That's the whole trip. Our running article changed form like this:

```text
source string
   ↓ byte-pair tokenizer (lowercase, split into pieces)
token pieces → token IDs + attention mask      [8, L], L ≤ 77
   ↓ text transformer (causal attention)
one contextual state per position              [8, L, 512]
   ↓ read the EOS position, project, normalize
one unit-length article vector                 [8, 512]
```

**The big ideas:**

- A tokenizer turns text into pieces from a fixed vocabulary. BPE keeps common words whole and splits rare ones.
- Token IDs are just addresses. Meaning only shows up once the transformer turns them into **contextual states**.
- Special tokens, padding, truncation, and the attention mask let articles of different lengths share one tensor, and the 77-position limit decides what the model can see at all.
- Because of the causal mask, only the **EOS position** has seen the whole article, so that's where CLIP reads its summary.
- After projection and normalization, text vectors live in the same 512-number space as image vectors.

**Tokenization decides which positions the model gets. The transformer decides how those positions inform each other.**

### Check your understanding

1. Token IDs 1000 and 1001 sit right next to each other in the vocabulary. Does that mean the model treats them as similar?
2. An article tokenizes into 120 pieces. With `truncation=True` and `max_length=77`, how many of its pieces does the model actually see?
3. On the vision side, CLIP read its summary from a class token at the front. Why does the text side read it from the EOS position at the end instead?

<details>
<summary>Show answers</summary>

1. No. An ID is just an address in a list. Any similarity comes from the learned vectors the model looks up and builds from those IDs, not from the numbers themselves.
2. 77 positions in total, but two of them are BOS and EOS. So the model sees the first **75** pieces, and the other 45 are dropped.
3. The text transformer uses a causal mask: each position only sees itself and earlier positions. EOS is the last real position, so it's the only one that has seen the whole article.

</details>

## Try it yourself

### TODO 1: Which study article is closest to your headline?

The cell below runs the whole text pipeline on your own sentences, then finds the closest of our eight study articles for each one. It runs as-is, but the fun part is writing your own headlines.

Before you run it, guess: which topic will each default headline land closest to?

In [ ]:
# TODO: swap in your own headlines (as many as you like)
my_headlines = [
    "Stocks rally as tech earnings beat expectations",
    "Underdog team wins the championship in overtime",
]
my_batch = tokenizer(my_headlines, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)

with torch.inference_mode():
    my_pooled = model.text_model(input_ids=my_batch["input_ids"], attention_mask=my_batch["attention_mask"]).pooler_output
    my_vectors = F.normalize(model.text_projection(my_pooled), dim=-1)

my_scores = my_vectors @ article_vectors.T  # [number of headlines, 8]
for headline, scores in zip(my_headlines, my_scores):
    best = scores.argmax().item()
    print(f"{headline[:50]:50s} -> closest: {labels[best]} ({scores[best].item():.3f})")

Try a headline that mixes two topics, like a sports team's business deal. Where does it land?

### TODO 2: Watch truncation eat an article

What happens to text past the 77-position limit? The cell below glues a few study articles together into one long text and compares the full piece count with what the model actually gets.

Before you run it, guess: how many positions will the model get?

In [ ]:
# TODO: change how many articles get glued together (1 to 8)
num_glued = 3
long_text = " ".join(study_texts[:num_glued])
long_pieces = tokenizer.tokenize(long_text)  # no limit here
long_ids = tokenizer(long_text, truncation=True, max_length=max_length)["input_ids"]  # what the model would get

print("pieces before truncation:", len(long_pieces))
print("positions the model gets:", len(long_ids))
print("last 200 characters it sees:", tokenizer.decode(long_ids, skip_special_tokens=True)[-200:])

<details>
<summary>Hint</summary>

The model never gets more than 77 positions, BOS and EOS included. Look at where the decoded text stops: it's probably in the middle of one of the later articles. Everything after that point is invisible to CLIP, which is one reason CLIP works best with short, caption-sized text.

</details>

### TODO 3: Does padding change an article's vector?

In our batch, shorter articles got padded. Does that padding sneak into their vectors? Let's find out: run one article **alone** (so there's no padding at all) and compare its vector with the one from the batch.

In [ ]:
# TODO: try other rows (0 to 7); rows with a short visible length get the most padding
solo_row = 0
solo_batch = tokenizer([study_texts[solo_row]], truncation=True, max_length=max_length, return_tensors="pt").to(device)

with torch.inference_mode():
    solo_pooled = model.text_model(input_ids=solo_batch["input_ids"], attention_mask=solo_batch["attention_mask"]).pooler_output
    solo_vector = F.normalize(model.text_projection(solo_pooled), dim=-1)

print("batch width:", token_batch["input_ids"].shape[1], "| solo width:", solo_batch["input_ids"].shape[1])
print("cosine(solo, batched):", round((solo_vector @ article_vectors[solo_row:solo_row + 1].T).item(), 4))

<details>
<summary>Hint</summary>

You should get 1.0, or something extremely close to it (tiny floating-point differences are normal). Two things keep padding out: the attention mask tells the model to ignore padding positions, and because of the causal mask, the EOS position can't look ahead at the padding anyway.

</details>

---

**Up next, Notebook 03:** we finally put the two halves together. We'll set image vectors from Notebook 01 next to text vectors like these and rebuild the rule CLIP uses to decide which caption goes with which image.